In [1]:
import os
import tensorflow as tf
import pandas as pd
import numpy as np

from tensorflow.keras import layers, models
from tensorflow.keras.applications import ResNet50

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))



TensorFlow: 2.10.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [2]:
from transformers import TFBertModel, BertTokenizer

c:\Users\sarth\anaconda3\envs\tf_prod_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
DATASET_DIR = "/notebook/data/"
CSV_PATH = DATASET_DIR + "amazon_multimodal_dataset_with_ratings.csv"
IMAGE_BASE_DIR = DATASET_DIR
CKPT_DIR = "/notebook/checkpoints/"

BATCH_SIZE = 32
MAX_LEN = 128
EMBED_DIM = 256
TOTAL_EPOCHS = 20




In [ ]:
df = pd.read_csv(CSV_PATH)
print("Rows:", len(df))
df.head()




Rows: 19601


,ASIN,image_path,review_text
0,B00IOHU8T4,images/B00IOHU8T4.jpg,Very nice
1,B00ARBGDJ4,images/B00ARBGDJ4.jpg,Cheap junk broke 1 hour into use! This is very...
2,B00B6E7IHW,images/B00B6E7IHW.jpg,The sizing and weight of the shirt are just as...
3,B001UQULU2,images/B001UQULU2.jpg,"I loved the movies, since I was a kid watching..."
4,B002JPJHKI,images/B002JPJHKI.jpg,The stitching on the sandals came apart after ...


In [ ]:
print(df.isnull().sum())
df=df.dropna(subset=["review_text"])
print(df.isnull().sum())

ASIN           0
image_path     0
review_text    3
dtype: int64
ASIN           0
image_path     0
review_text    0
dtype: int64


In [ ]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

In [ ]:
def load_image_tf(path):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, (224, 224))
    img = tf.cast(img, tf.float32) / 255.0
    return img


In [ ]:
def tokenize_py(text):
    tokens = tokenizer(
        text.numpy().decode("utf-8"),
        padding="max_length",
        truncation=True,
        max_length=MAX_LEN,
        return_tensors="np"
    )
    return tokens["input_ids"][0], tokens["attention_mask"][0]


In [ ]:
def process_sample(image_path, review_text):
    # Image
    full_path = tf.strings.join([IMAGE_BASE_DIR, image_path])
    image = load_image_tf(full_path)

    # Text (Python tokenizer via py_function)
    input_ids, attention_mask = tf.py_function(
        tokenize_py,
        [review_text],
        [tf.int32, tf.int32]
    )

    # IMPORTANT: set static shapes
    input_ids.set_shape([MAX_LEN])
    attention_mask.set_shape([MAX_LEN])

    return image, input_ids, attention_mask




In [ ]:
dataset = tf.data.Dataset.from_tensor_slices(
    (df["image_path"].values, df["review_text"].values)
)

dataset = (
    dataset
    .shuffle(1000)
    .map(process_sample, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)



In [ ]:
def build_image_encoder(embed_dim=256):
    inputs = layers.Input(shape=(224, 224, 3))

    base = ResNet50(
        weights="imagenet",
        include_top=False,
        input_tensor=inputs
    )
    base.trainable = False

    x = base.output
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(embed_dim)(x)

    outputs = layers.Lambda(
        lambda t: tf.math.l2_normalize(t, axis=1),
        output_shape=(embed_dim,)
    )(x)

    return models.Model(inputs, outputs)


In [ ]:
def build_text_encoder(embed_dim=256):
    bert = TFBertModel.from_pretrained(
        "bert-base-uncased",
        from_pt=True
    )
    bert.trainable = False

    input_ids = layers.Input(shape=(MAX_LEN,), dtype=tf.int32)
    attention_mask = layers.Input(shape=(MAX_LEN,), dtype=tf.int32)

    def bert_call(inputs):
        ids, mask = inputs
        out = bert(input_ids=ids, attention_mask=mask)
        return out.last_hidden_state[:, 0, :]  # CLS token

    cls_emb = layers.Lambda(
        bert_call,
        output_shape=(768,)
    )([input_ids, attention_mask])

    x = layers.Dense(embed_dim)(cls_emb)

    outputs = layers.Lambda(
        lambda t: tf.math.l2_normalize(t, axis=1),
        output_shape=(embed_dim,)
    )(x)

    return models.Model([input_ids, attention_mask], outputs)


In [ ]:
image_encoder = build_image_encoder(EMBED_DIM)
text_encoder = build_text_encoder(EMBED_DIM)

optimizer = tf.keras.optimizers.Adam(1e-4)


94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

TensorFlow and JAX classes are deprecated and will be removed in Transformers v5. We recommend migrating to PyTorch classes or pinning your version of Transformers.
Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFBertModel: ['cls.predictions.transform.LayerNorm.bias', 'cls.seq_relationship.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.weight', 'cls.seq_relationship.weight', 'cls.predictions.bias', 'cls.predictions.decoder.weight', 'cls.predictions.transform.dense.bias']
- This IS expected if you are initializing TFBertModel from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFBertModel from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
Al

In [ ]:
class ContrastiveLoss(tf.keras.losses.Loss):
    def __init__(self, temperature=0.07):
        super().__init__()
        self.temperature = temperature

    def call(self, img_emb, txt_emb):
        logits = tf.matmul(img_emb, txt_emb, transpose_b=True)
        logits /= self.temperature
        labels = tf.range(tf.shape(logits)[0])
        return tf.reduce_mean(
            tf.keras.losses.sparse_categorical_crossentropy(
                labels, logits, from_logits=True
            )
        )

loss_fn = ContrastiveLoss()


In [ ]:
@tf.function(reduce_retracing=True)
def train_step(images, input_ids, attention_mask):
    with tf.GradientTape() as tape:
        img_emb = image_encoder(images, training=True)
        txt_emb = text_encoder([input_ids, attention_mask], training=False)
        loss = loss_fn(img_emb, txt_emb)

    vars = image_encoder.trainable_variables
    grads = tape.gradient(loss, vars)
    optimizer.apply_gradients(zip(grads, vars))
    return loss



In [ ]:
checkpoint = tf.train.Checkpoint(
    image_encoder=image_encoder,
    text_encoder=text_encoder,
    optimizer=optimizer
)

ckpt_manager = tf.train.CheckpointManager(
    checkpoint,
    CKPT_DIR,
    max_to_keep=5
)

if ckpt_manager.latest_checkpoint:
    checkpoint.restore(ckpt_manager.latest_checkpoint)
    print("✅ Restored from:", ckpt_manager.latest_checkpoint)
else:
    print("🆕 Training from scratch")


✅ Restored from: /content/drive/MyDrive/Datasets/checkpoints/ckpt-7


In [ ]:
for epoch in range(TOTAL_EPOCHS):
    print(f"\nEpoch {epoch+1}/{TOTAL_EPOCHS}")

    for images, input_ids, attention_mask in dataset:
        loss = train_step(images, input_ids, attention_mask)

    save_path = ckpt_manager.save()
    print(f"💾 Saved checkpoint: {save_path}")
    print("Epoch loss:", loss.numpy())



Epoch 1/15
💾 Saved checkpoint: /content/drive/MyDrive/Datasets/checkpoints/ckpt-8
Epoch loss: 2.1302428

Epoch 2/15
💾 Saved checkpoint: /content/drive/MyDrive/Datasets/checkpoints/ckpt-9
Epoch loss: 2.3872952

Epoch 3/15
💾 Saved checkpoint: /content/drive/MyDrive/Datasets/checkpoints/ckpt-10
Epoch loss: 2.0313823

Epoch 4/15
💾 Saved checkpoint: /content/drive/MyDrive/Datasets/checkpoints/ckpt-11
Epoch loss: 2.2296078

Epoch 5/15
💾 Saved checkpoint: /content/drive/MyDrive/Datasets/checkpoints/ckpt-12
Epoch loss: 2.1633153

Epoch 6/15
💾 Saved checkpoint: /content/drive/MyDrive/Datasets/checkpoints/ckpt-13
Epoch loss: 2.2646923

Epoch 7/15
💾 Saved checkpoint: /content/drive/MyDrive/Datasets/checkpoints/ckpt-14
Epoch loss: 2.3292444

Epoch 8/15
💾 Saved checkpoint: /content/drive/MyDrive/Datasets/checkpoints/ckpt-15
Epoch loss: 1.8088161

Epoch 9/15
💾 Saved checkpoint: /content/drive/MyDrive/Datasets/checkpoints/ckpt-16
Epoch loss: 2.0506926

Epoch 10/15
💾 Saved checkpoint: /content/drive/

In [ ]:
import os

FINAL_MODEL_DIR = "/content/drive/MyDrive/Datasets/final_models"
os.makedirs(FINAL_MODEL_DIR, exist_ok=True)


In [ ]:
IMAGE_MODEL_PATH = "/content/drive/MyDrive/Datasets/final_models/image_encoder.keras"
image_encoder.save(IMAGE_MODEL_PATH)



In [ ]:
TEXT_MODEL_DIR = "/content/drive/MyDrive/Datasets/final_models/text_encoder.keras"
text_encoder.save(TEXT_MODEL_DIR)


In [ ]:
print("Latest checkpoint:", ckpt_manager.latest_checkpoint)


Latest checkpoint: /content/drive/MyDrive/Datasets/checkpoints/ckpt-22


In [ ]:
import json

config = {
    "model": "ResNet50 + BERT (contrastive)",
    "image_size": [224, 224],
    "max_len": MAX_LEN,
    "embed_dim": EMBED_DIM,
    "batch_size": BATCH_SIZE,
    "epochs": TOTAL_EPOCHS,
    "dataset": "Amazon multimodal (images + reviews)",
    "loss": "Contrastive (InfoNCE)"
}

with open("/content/drive/MyDrive/Datasets/final_models/config.json", "w") as f:
    json.dump(config, f, indent=2)


#creating the prediction data ready


In [4]:
import tensorflow as tf
import numpy as np
import pandas as pd
from transformers import BertTokenizer


In [5]:
df = pd.read_csv(
    "/content/drive/MyDrive/Datasets/amazon_multimodal_dataset_with_ratings.csv"
)

df = df.dropna(subset=["review_text", "rating"]).reset_index(drop=True)

print("Total reviews:", len(df))

Total reviews: 19600


In [6]:
def build_text_encoder(embed_dim=256):
    bert = TFBertModel.from_pretrained(
        "bert-base-uncased",
        from_pt=True
    )
    bert.trainable = False

    input_ids = layers.Input(shape=(MAX_LEN,), dtype=tf.int32)
    attention_mask = layers.Input(shape=(MAX_LEN,), dtype=tf.int32)

    def bert_call(inputs):
        ids, mask = inputs
        out = bert(input_ids=ids, attention_mask=mask)
        return out.last_hidden_state[:, 0, :]  # CLS token

    cls_emb = layers.Lambda(
        bert_call,
        output_shape=(768,)
    )([input_ids, attention_mask])

    x = layers.Dense(embed_dim)(cls_emb)

    outputs = layers.Lambda(
        lambda t: tf.math.l2_normalize(t, axis=1),
        output_shape=(embed_dim,)
    )(x)

    return models.Model([input_ids, attention_mask], outputs)




In [7]:
text_encoder = build_text_encoder(EMBED_DIM)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

TensorFlow and JAX classes are deprecated and will be removed in Transformers v5. We recommend migrating to PyTorch classes or pinning your version of Transformers.
Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFBertModel: ['cls.predictions.bias', 'cls.seq_relationship.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.seq_relationship.bias', 'cls.predictions.transform.dense.weight', 'cls.predictions.decoder.weight', 'cls.predictions.transform.dense.bias']
- This IS expected if you are initializing TFBertModel from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFBertModel from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
Al

In [8]:
ckpt = tf.train.Checkpoint(text_encoder=text_encoder)

ckpt.restore(
    tf.train.latest_checkpoint(
        "/content/drive/MyDrive/Datasets/checkpoints"
    )
).expect_partial()

print("✅ Text encoder weights restored correctly")


KeyboardInterrupt: 

In [ ]:
MAX_LEN = 128
BATCH_SIZE = 32

def encode_reviews(texts):
    all_embeddings = []

    for i in range(0, len(texts), BATCH_SIZE):
        batch_texts = texts[i:i+BATCH_SIZE].tolist()

        tokens = tokenizer(
            batch_texts,
            padding="max_length",
            truncation=True,
            max_length=MAX_LEN,
            return_tensors="tf"
        )

        emb = text_encoder(
            [tokens["input_ids"], tokens["attention_mask"]],
            training=False
        )

        all_embeddings.append(emb.numpy())

    return np.vstack(all_embeddings)

In [ ]:
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")


In [ ]:
review_embeddings = encode_reviews(df["review_text"])

np.save(
    "/content/drive/MyDrive/Datasets/final_models/review_embeddings.npy",
    review_embeddings
)


In [6]:
import os
os.path.exists("/content/drive/MyDrive/Datasets/final_models/review_embeddings.npy")


True

In [7]:
from keras.saving import register_keras_serializable
from tensorflow.keras import layers

@register_keras_serializable()
class L2Normalize(layers.Layer):
    def call(self, inputs):
        return tf.math.l2_normalize(inputs, axis=1)


In [8]:
from tensorflow.keras.applications import ResNet50
from tensorflow.keras import models

def build_image_encoder(embed_dim=256):
    inputs = layers.Input(shape=(224, 224, 3))

    base = ResNet50(
        weights="imagenet",
        include_top=False,
        input_tensor=inputs
    )
    base.trainable = False

    x = base.output
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(embed_dim)(x)
    outputs = L2Normalize()(x)

    return models.Model(inputs, outputs)


In [9]:
image_encoder = build_image_encoder(EMBED_DIM)

ckpt = tf.train.Checkpoint(image_encoder=image_encoder)
ckpt.restore(
    tf.train.latest_checkpoint(
        "/content/drive/MyDrive/Datasets/checkpoints"
    )
).expect_partial()

print("✅ Image encoder weights restored from checkpoint")


✅ Image encoder weights restored from checkpoint


In [10]:
image_encoder.save(
    "/content/drive/MyDrive/Datasets/final_models/image_encoder_safe.keras"
)


In [11]:
import numpy as np
import pandas as pd
import tensorflow as tf


image_encoder = tf.keras.models.load_model(
    "/content/drive/MyDrive/Datasets/final_models/image_encoder_safe.keras"
)

review_embeddings = np.load(
    "/content/drive/MyDrive/Datasets/final_models/review_embeddings.npy"
)

df = pd.read_csv(
    "/content/drive/MyDrive/Datasets/amazon_multimodal_dataset_with_ratings.csv"
).dropna(subset=["review_text", "rating"]).reset_index(drop=True)



In [12]:
def encode_image(image_path):
    img = tf.io.read_file(image_path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, (224, 224))
    img = tf.cast(img, tf.float32) / 255.0
    img = tf.expand_dims(img, axis=0)

    emb = image_encoder(img, training=False)
    return emb.numpy()[0]


In [13]:
def get_top_k(image_embedding, k=5):
    # cosine similarity (embeddings are normalized)
    scores = np.dot(review_embeddings, image_embedding)

    top_k_idx = np.argsort(scores)[-k:][::-1]
    return df.iloc[top_k_idx], scores[top_k_idx]


In [14]:
def compute_final_rating(top_reviews, similarities, min_sim=0.7):
    ratings = top_reviews["rating"].values.astype(float)

    # Keep only sufficiently similar reviews
    mask = similarities >= min_sim

    if mask.sum() == 0:
        # fallback: mean of all top-k
        return float(np.mean(ratings))

    ratings = ratings[mask]
    similarities = similarities[mask]

    # Weighted average
    weights = similarities
    rating = np.sum(ratings * weights) / np.sum(weights)

    return float(rating)



In [15]:
def predict_rating_from_image(image_path, k=10, min_sim=0.7):
    img_emb = encode_image(image_path)
    top_reviews, sims = get_top_k(img_emb, k)
    rating = compute_final_rating(top_reviews, sims, min_sim)
    return round(rating, 2)



In [16]:
image_path = "/content/drive/MyDrive/Datasets/images/B00ARBGDJ4.jpg"

predicted_rating = predict_rating_from_image(image_path, k=10)

print("⭐ Predicted Rating:", predicted_rating)


⭐ Predicted Rating: 4.6


In [17]:
from sklearn.model_selection import train_test_split

eval_df = df.sample(1000, random_state=42)  # or separate test set


In [18]:
y_true = []
y_pred = []

for _, row in eval_df.iterrows():
    image_path = IMAGE_BASE_DIR + row["image_path"]
    true_rating = float(row["rating"])

    try:
        pred_rating = predict_rating_from_image(image_path, k=10)
        y_true.append(true_rating)
        y_pred.append(pred_rating)
    except:
        continue


In [19]:
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error

y_true = np.array(y_true)
y_pred = np.array(y_pred)

mae = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))

print("MAE:", round(mae, 3))
print("RMSE:", round(rmse, 3))


MAE: 1.003
RMSE: 1.299


In [20]:
within_05 = np.mean(np.abs(y_true - y_pred) <= 0.5)
within_10 = np.mean(np.abs(y_true - y_pred) <= 1.0)

print("Accuracy ±0.5 stars:", round(within_05 * 100, 2), "%")
print("Accuracy ±1.0 stars:", round(within_10 * 100, 2), "%")


Accuracy ±0.5 stars: 31.9 %
Accuracy ±1.0 stars: 65.7 %
